# T06 — Codificación posicional

## 1. Título y paper

**Paper:** *Attention Is All You Need* (Vaswani et al., 2017)  
**Fuente primaria:** [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)  
**Foco de esta miniatura:** la atención es permutación-equivariante y eso es un problema  
**Ficha completa:** [`P08_transformer`](../../papers/foundational/P08_transformer/README.md)


## 2. Objetivos

1. Demostrar que sin posición, «el gato come» y «come gato el» son idénticos para la atención.
2. Verificar las propiedades de la codificación sinusoidal.


## 3. Prerrequisitos

- Python 3.11+ con el paquete instalado (`pip install -e .`).
- Notebook [`P08_transformer`](P08_transformer.ipynb) al menos hojeado.
- Álgebra de vectores: producto escalar, norma y softmax.


## 4. Intuición

La atención es un conjunto, no una lista: si barajas los tokens, la salida se baraja igual pero no cambia. Hay que inyectar la posición en el propio vector.


## 5. Concepto mínimo

```text
PE(pos, 2i)   = sin(pos / 10000^{2i/d_model})
PE(pos, 2i+1) = cos(pos / 10000^{2i/d_model})
```

Se **suma** al embedding. Las frecuencias distintas por dimensión dan una firma única por posición y permiten extrapolar a longitudes no vistas en entrenamiento.


## 6. Código explicado

Código mínimo, sin dependencias externas.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
from ai_evolution.papers_lab import positional_encoding, scaled_dot_product_attention

A = [[1.0, 0.0], [0.0, 1.0], [0.5, 0.5]]
B = [A[2], A[0], A[1]]                     # misma bolsa de tokens, otro orden
sa = scaled_dot_product_attention(A, A, A)['output']
sb = scaled_dot_product_attention(B, B, B)['output']
print('salida A :', [[round(v, 4) for v in f] for f in sa])
print('salida B :', [[round(v, 4) for v in f] for f in sb])
print('¿es B una permutación de A?', sorted(map(str, sa)) == sorted(map(str, sb)))

## 7. Predicción antes de ejecutar

Si sumamos PE a cada token, ¿seguirá siendo la salida de B una permutación de la de A?

> Escribe tu respuesta antes de continuar.


## 8. Experimento controlado


In [ ]:
def con_posicion(seq):
    return [[v + p for v, p in zip(tok, positional_encoding(i, len(tok)))]
            for i, tok in enumerate(seq)]

sa2 = scaled_dot_product_attention(*[con_posicion(A)] * 3)['output']
sb2 = scaled_dot_product_attention(*[con_posicion(B)] * 3)['output']
print('con PE, A:', [[round(v, 4) for v in f] for f in sa2])
print('con PE, B:', [[round(v, 4) for v in f] for f in sb2])
print('¿siguen siendo permutación una de otra?',
      sorted(map(str, sa2)) == sorted(map(str, sb2)))

## 9. Salida interpretable

Sin PE, reordenar la entrada solo reordena la salida: el modelo es ciego al orden. Con PE, las salidas dejan de ser permutaciones entre sí: **el orden ya es información**.


## 10. Comentario pedagógico

Esta miniatura aísla **una** pieza del bloque. Aislar es didáctico y también es una simplificación: en el modelo real todas las piezas interactúan y se entrenan juntas.


## 11. Error o anti-patrón deliberado


In [ ]:
print('Error: CONCATENAR la posición en vez de sumarla, «para no contaminar el embedding».')
print('Concatenar cambia d_model, multiplica los parámetros de todas las proyecciones')
print('y rompe la compatibilidad con las conexiones residuales, que exigen misma dimensión.')

## 12. Corrección


In [ ]:
emb = [0.4, -0.2, 0.1, 0.7]
pe = positional_encoding(3, 4)
print('embedding:', emb)
print('PE(pos=3):', [round(v, 4) for v in pe])
print('suma     :', [round(a + b, 4) for a, b in zip(emb, pe)], '· dimensión intacta:', len(emb))

## 13. Desafío guiado

Comprueba si PE(pos+k) se puede expresar como una transformación lineal de PE(pos) para k fijo.


## 14. Desafío autónomo

Reescribe esta pieza con proyecciones aprendidas y comprueba que tu implementación reproduce las propiedades verificadas aquí (sumas, formas, invariantes). Documenta la semilla.


## 15. Evidencia de aprendizaje

Guarda la salida del experimento, tu predicción previa y una frase sobre qué invariante acabas de verificar.


## 16. Cierre

Pieza cubierta: **la atención es permutación-equivariante y eso es un problema**. Ya puede describirse con precisión, sin metáforas.


## 17. Conexión con el siguiente hito

Con posición y atención resueltas, falta el andamiaje que permite apilar capas: residual y layer norm (T07).
